我先快速读取你给的 `ETA-Resource` 仓库首页说明，确认原始 pcap 数据结构，然后给你一套“从下载数据到按脚本全链路执行”的分阶段方案（含每个脚本顺序、输入输出、检查点和可跳过项）。



Fetched https://github.com/linwhitehat/ETA-Resource.git

我再补一次全量脚本清单并按可执行性分组，确保“尽量完整跑一遍所有文件”的方案不漏脚本、且能按依赖顺序执行。

搜索与 `**/costSensitive/**/*.py` 匹配的文件，32 匹配项

你这个需求很专业，我给你一个“**全链路+尽量覆盖所有脚本**”的执行方案。  
先说关键事实：**ETA-Resource 是资源导航仓库，不直接提供单一可下载数据集**，你需要从其中某个数据集链接另行下载 pcap。

**总体目标**
- 从原始 pcap 开始，走完：`pcap -> packet/png -> idx(gz) -> 训练 -> 推理 -> 评估日志`
- 在不破坏主线的前提下，尽量执行其余脚本（含历史脚本、RSG 子工程）

**阶段 0：环境与基线**
- 进入项目根目录：`costSensitive-new`
- 建议 Python 3.10 虚拟环境，安装：`torch torchvision numpy pandas matplotlib scikit-learn pillow opencv-python scapy dpkt gevent`
- 基线先确认主线可跑（你已完成）：RSG_pytroch_CNN784.py + inference.py

**阶段 1：准备 pcap 数据（来自 ETA-Resource 指向的数据源）**
- 从 https://github.com/linwhitehat/ETA-Resource.git 里挑一个“可获取原始 pcap”的数据集下载
- 把数据整理成以下目录之一（按脚本硬编码）：
  - costSensitive/nonvpn_dataset（给 PcaptoImage.py）
  - costSensitive/vpn_dataset（给 PcaptoImage-vpn-all.py）
  - 10lei（给 PcaptoImage2.py）
- 每个类别一个子目录，目录里放 pcap/pcapng 文件

**阶段 2：pcap 预处理到 png（多脚本分支，建议逐个跑）**
- 非 VPN 流程：`python costSensitive/PcaptoImage.py`
- VPN 流程：`python costSensitive/PcaptoImage-vpn-all.py`
- 10 类流程：`python costSensitive/PcaptoImage2.py`
- data_processing 变体：`python costSensitive/data_processing/PcaptoImage-nonvpn-all.py`
- 产物会落到 packet-nonvpn、packet-vpn、png-10、png-nonvpn、png-vpn

**阶段 3：png 转 IDX（训练数据格式）**
- 先按 mnist.py 中 `Names` 配置确认源目录（默认是 `png-10/Train|Test`）
- 运行：`python costSensitive/mnist.py`
- 产物在 costSensitive/5_Mnist-10（`*-idx*-ubyte` + `.gz`）

**阶段 4：数据检查脚本（建议都跑）**
- `python costSensitive/fenxi.py`（解析 IDX 并可视化）
- `python costSensitive/analysis.py`（各类数据量柱状图）
- `python costSensitive/copypng.py`（抽样拆分；注意它是 move，不是 copy，先备份）
- `python costSensitive/test.py`（张量逻辑小测试）

**阶段 5：训练与推理（PyTorch 主线）**
- 训练：`python costSensitive/RSG_pytroch_CNN784.py`
- 推理：`python costSensitive/inference.py`
- 检查产物：
  - convnet.pth
  - train_log.csv
  - predictions.csv

**阶段 6：历史/扩展脚本尽量覆盖**
- 旧 PyTorch 版本：`python costSensitive/cnn784-pytorch.py`（注意它默认 12 类头，需与数据类别匹配）
- 旧 TensorFlow 路线：`python costSensitive/cnn784.py`（需 TF1 兼容环境，不建议与主线同环境混跑）
- RSG 子工程（能完整跑的最小入口）：
  - 进入 Imbalanced_Classification
  - 跑 CIFAR：`python cifar_train.py --dataset cifar10 --epochs 2 --imb_type exp --imb_factor 0.01 --loss_type CE --train_rule None`
  - 其余 `imagenet_lt_* / places_* / inaturalist_*` 需要额外大规模数据，属于可选扩展

**阶段 7：建议的“全量执行顺序”**
- 1) PcaptoImage 系列（4个）  
- 2) mnist.py  
- 3) fenxi / analysis / copypng / test  
- 4) RSG_pytroch_CNN784.py  
- 5) inference.py  
- 6) cnn784-pytorch / cnn784（旧线）  
- 7) RSGmain 的 cifar_train，再决定是否上大数据集脚本

如果你要，我可以下一步直接给你一份“**Windows 一键执行脚本**”（按这个顺序、每步自动检查输入输出是否存在，失败就停并提示修复点）。